# KW — CSV Ingestion

Reads every CSV under `Files/KW/raw/` and loads it into the Delta tables created by `01_create_schema.ipynb`.

**Prereq**: Upload the contents of `data/` to `Files/KW/raw/` in the `KW_lh` Lakehouse (drag-drop in the Lakehouse explorer, or use OneLake file explorer).

In [ ]:
LAKEHOUSE = 'KW_lh'
RAW_PATH  = f'Files/KW/raw'
spark.sql(f'USE {LAKEHOUSE}')

In [ ]:
# Map: csv file (without extension) -> target Delta table name
MAPPING = {
    # dimensions
    'warehouses':          'dim_warehouse',
    'stores':              'dim_store',
    'fridges_freezers':    'dim_fridge',
    'suppliers':           'dim_supplier',
    'products':            'dim_product',
    'trucks':              'dim_truck',
    'drivers':             'dim_driver',
    'routes':              'dim_route',
    'forklifts':           'dim_forklift',
    'employees':           'dim_employee',
    'customers':           'dim_customer',
    'campaigns':           'dim_campaign',
    'promotions':          'dim_promotion',
    # facts
    'store_stock':         'fact_store_stock',
    'inwards_goods':       'fact_inwards_goods',
    'shipments':           'fact_shipments',
    'shipment_lines':      'fact_shipment_lines',
    'pick_events':         'fact_pick_events',
    'trips':               'fact_trips',
    'sales':               'fact_sales',
    'sale_lines':          'fact_sale_lines',
    'employee_skills':     'fact_employee_skills',
    'performance_kpis':    'fact_performance_kpis',
    'training_records':    'fact_training_records',
    'campaign_performance':'fact_campaign_performance',
    'energy_consumption':  'fact_energy_consumption',
    'wastage':             'fact_wastage',
}

In [ ]:
from pyspark.sql.utils import AnalysisException

def load(csv_name: str, table_name: str):
    path = f'{RAW_PATH}/{csv_name}.csv'
    print(f'\nLoading {path} -> {table_name}')
    df = (spark.read
          .option('header', True)
          .option('inferSchema', True)
          .option('multiLine', True)
          .csv(path))
    target_schema = spark.table(table_name).schema
    # cast to target schema column-by-column where possible
    for f in target_schema.fields:
        if f.name in df.columns:
            df = df.withColumn(f.name, df[f.name].cast(f.dataType))
    df = df.select(*[f.name for f in target_schema.fields if f.name in df.columns])
    df.write.mode('overwrite').option('mergeSchema', True).saveAsTable(table_name)
    print(f'  loaded {df.count():,} rows')

for csv_name, table_name in MAPPING.items():
    try:
        load(csv_name, table_name)
    except AnalysisException as e:
        print(f'  SKIPPED {csv_name}: {e}')

## Validate row counts and sample data

In [ ]:
for t in MAPPING.values():
    n = spark.table(t).count()
    print(f'{t:40s} {n:>10,}')

In [ ]:
# Quick sanity: top selling products last 30 days
display(spark.sql("""
    SELECT p.name, p.category, sum(sl.qty) AS units, round(sum(sl.line_total),2) AS revenue
    FROM fact_sale_lines sl
    JOIN fact_sales s     ON s.sale_id = sl.sale_id
    JOIN dim_product p    ON p.product_id = sl.product_id
    WHERE s.sale_date >= date_sub(current_date(), 30)
    GROUP BY p.name, p.category
    ORDER BY revenue DESC
    LIMIT 15
"""))

## OPTIMIZE & Z-ORDER hot tables

In [ ]:
for t, cols in [
    ('fact_sales',         'store_id, sale_date'),
    ('fact_sale_lines',    'sale_id, product_id'),
    ('fact_pick_events',   'warehouse_id, pick_date'),
    ('fact_shipments',     'store_id, ship_date'),
    ('fact_inwards_goods', 'warehouse_id, receipt_date'),
]:
    spark.sql(f'OPTIMIZE {t} ZORDER BY ({cols})')
print('Optimized.')